# Talk to Your Database — Colab UI (Qwen2.5)

Runtime: **GPU (T4)**. Run cells **0 → 1 → 2 → 3 → 4** in order.

Cell 0 obtains the repository (git clone or zip), then locates the project root via `app/app.py`.

Cell 4 prints a Colab proxy URL — open that link (the embedded iframe may stay blank).

Do not use local Windows `localhost:8501` or Streamlit External URL for this session.

Sidebar should show live backend `qwen2.5-1.5b`.


## Cell 0 — Obtain project


In [ ]:
import os, shutil
from pathlib import Path

# --- Option A: GitHub clone (set your repo URL) ---
REPO_URL = ""  # e.g. "https://github.com/<org>/<repo>.git"
BRANCH = "main"

# --- Option B: zip upload (archive whose extracted tree contains app/) ---
ZIP_CANDIDATES = [
    "/content/sample_data/project.zip",
    "/content/project.zip",
]

def find_project_root(base=Path("/content")) -> Path:
    hits = sorted(base.glob("**/app/app.py"))
    hits = [h for h in hits if "/.git/" not in str(h).replace("\\", "/")]
    if not hits:
        raise FileNotFoundError("Could not find app/app.py under /content")
    return hits[0].resolve().parents[1]

if REPO_URL.strip():
    dest = Path("/content/repo")
    if dest.exists():
        shutil.rmtree(dest)
    get_ipython().system(f'git clone --depth 1 -b {BRANCH} "{REPO_URL}" /content/repo')
    print("Cloned", REPO_URL)
else:
    zip_path = next((z for z in ZIP_CANDIDATES if os.path.isfile(z)), None)
    if zip_path is None:
        for folder in (Path("/content/sample_data"), Path("/content")):
            zips = sorted(folder.glob("*.zip"))
            if zips:
                zip_path = str(zips[0])
                break
    if not zip_path:
        raise FileNotFoundError(
            "Set REPO_URL or upload a zip containing app/ to /content/ or /content/sample_data/"
        )
    get_ipython().system(f'unzip -q -o "{zip_path}" -d /content/')
    print("Unzipped", zip_path)

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
print("PROJECT_ROOT =", ROOT)

assert (ROOT / "app" / "backend" / "clarify.py").is_file(), "app/backend missing"
text = (ROOT / "app" / "backend" / "clarify.py").read_text(encoding="utf-8")
assert 'CLARIFY_VERSION = "schema-aware-v2"' in text, "Unexpected clarify.py revision"
print("Clarification module: schema-aware-v2")


## Cell 1 — Setup

In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/repo"))
if not (ROOT / "app" / "app.py").exists():
    hits = sorted(Path("/content").glob("**/app/app.py"))
    ROOT = hits[0].parents[1]
    os.environ["PROJECT_ROOT"] = str(ROOT)
os.chdir(ROOT)
print("cwd", ROOT)
get_ipython().system("pip install -q -r app/scripts/colab-ui-requirements.txt")

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4"
print("CUDA OK:", torch.cuda.get_device_name(0))
get_ipython().system("ls app/app.py")


## Cell 2 — Demo databases

In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/repo"))
os.chdir(ROOT)
get_ipython().system("python app/scripts/download_demo_databases.py")
get_ipython().system("ls demo_databases/*.sqlite")


## Cell 3 — Start Streamlit

Wait until the cell reports that Streamlit is ready, then run Cell 4.


In [ ]:
import os
import shutil
import socket
import time
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"]) if os.environ.get("PROJECT_ROOT") else Path("/content/repo")
if not (ROOT / "app" / "app.py").exists():
    ROOT = sorted(Path("/content").glob("**/app/app.py"))[0].parents[1]
os.chdir(ROOT)
print("Using", ROOT)

example = ROOT / "app" / "ui_config.qwen25.example.json"
shutil.copy2(example, ROOT / "app" / "ui_config.json")
print("ui_config <- qwen2.5-1.5b")

cfg_dir = ROOT / ".streamlit"
cfg_dir.mkdir(exist_ok=True)
(cfg_dir / "config.toml").write_text(
    "\n".join([
        "[server]",
        "headless = true",
        "enableCORS = false",
        "enableXsrfProtection = false",
        "port = 8501",
        'address = "0.0.0.0"',
        "",
        "[browser]",
        "gatherUsageStats = false",
        "",
    ]),
    encoding="utf-8",
)
print("Wrote", cfg_dir / "config.toml")

os.system("fuser -k 8501/tcp >/dev/null 2>&1")
time.sleep(1)

cmd = (
    f"cd {ROOT} && "
    "MODEL_BACKEND=qwen2.5-1.5b MODEL_SLUG=qwen2.5-coder-1.5b-instruct "
    "nohup python -m streamlit run app/app.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--browser.gatherUsageStats false "
    "> /tmp/streamlit_ui.log 2>&1 &"
)
get_ipython().system_raw(cmd)
print("Streamlit launching (CORS flags + config.toml).")

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

ok = False
for i in range(60):
    if port_open():
        ok = True
        break
    time.sleep(1)
    if i % 10 == 9:
        print(f"  waiting… {i+1}s")

if not ok:
    print("Port 8501 did not open. Log:")
    get_ipython().system("tail -n 40 /tmp/streamlit_ui.log")
else:
    print("Streamlit is ready (running in the background).")
    print("Next: run Cell 4 and open the printed Colab proxy URL.")


## Cell 4 — Open UI (click the link)

`serve_kernel_port_as_iframe` often shows **nothing**. This cell prints a **Colab proxy URL** — open it.

In [ ]:
import socket
import time
from IPython.display import display, HTML
from google.colab.output import eval_js

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

if not port_open():
    raise RuntimeError("Port 8501 is closed. Re-run Cell 3 until READY, then this cell.")

time.sleep(1)
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("Expected sidebar backend: qwen2.5-1.5b")
print("Colab proxy URL:")
print(url)
display(HTML(f'<p><a href="{url}" target="_blank" style="font-size:18px">Open Talk to Your Database</a></p>'))
display(HTML(
    f'<iframe src="{url}" width="100%" height="800" style="border:1px solid #ccc"></iframe>'
))

## Optional — Stop Streamlit

In [ ]:
import os
os.system("fuser -k 8501/tcp >/dev/null 2>&1")
print("Stopped processes on port 8501 (if any).")